# Stage 1: initial-stance training examples

The model speaks as the tracked participant and reports that person's **initial** agreement with the proposition, on a 0–100 scale.

Each example contains only:

- a system turn with the proposition
- an assistant turn with the participant's own initial text, when that text exists
- a user turn asking for the 0–100 answer

Discussion turns, the final stance, the final context, and outcome fields are not part of these examples. CMV targets are the binary proxy `100` (holds the view) on a 0–100 scale and should be left out of regression metrics.

Apply a tokenizer with `tokenizer.apply_chat_template(messages, add_generation_prompt=True)`. The completion after that prompt is the only rewarded turn.

## Load

In [1]:
from pathlib import Path
import json
from collections import Counter

from opinion_dynamics.tasks.initial_stance import InitialStanceDataLoader, USER_QUESTION

REPO = Path('..').resolve()
if not (REPO / 'configs' / 'datasets.yaml').exists():
    REPO = Path('.').resolve()

loader = InitialStanceDataLoader.from_config(REPO / 'configs' / 'datasets.yaml', repo_root=REPO)
examples = loader.load()
summary = loader.summary(examples)
print(json.dumps({k: summary[k] for k in summary if k != 'target_histograms'}, indent=2))

{
  "n_examples": 10774,
  "per_dataset": {
    "AnthropicPersuasion": 3939,
    "CMV": 2931,
    "ConspiracyAI": 1749,
    "DEBATE": 1052,
    "DebateGPT": 750,
    "PersuasionTrace": 353
  },
  "per_split": {
    "test": 1369,
    "train": 8281,
    "val": 1124
  },
  "with_initial_context": 5732,
  "without_initial_context": 5042,
  "binary_proxy": 2931,
  "exclusions": {
    "CMV post contains a delta trace": 183,
    "duplicate CMV post; kept one episode": 1126
  }
}


## Counts and target histograms

In [2]:
for name, hist in summary['target_histograms'].items():
    top = sorted(hist.items(), key=lambda item: -item[1])[:8]
    print(f'{name:22} n={summary["per_dataset"][name]:5d}  top targets={top}')
print('exclusions', summary['exclusions'])
cmv = [ex for ex in examples if ex.dataset == 'CMV']
assert cmv and {ex.target for ex in cmv} == {100.0}
assert len({ex.group_id for ex in cmv}) == len(cmv)
print(f'CMV posts {len(cmv)} (one episode each, target always 100)')

AnthropicPersuasion    n= 3939  top targets=[(50.0, 706), (83.33333333333334, 671), (66.66666666666666, 630), (16.666666666666664, 579), (33.33333333333333, 534), (100.0, 491), (0.0, 328)]
CMV                    n= 2931  top targets=[(100.0, 2931)]
ConspiracyAI           n= 1749  top targets=[(100.0, 423), (80.0, 108), (90.0, 95), (60.0, 55), (70.0, 52), (75.0, 52), (50.0, 51), (81.0, 47)]
DEBATE                 n= 1052  top targets=[(40.0, 236), (100.0, 226), (80.0, 204), (20.0, 130), (60.0, 128), (0.0, 128)]
DebateGPT              n=  750  top targets=[(75.0, 186), (100.0, 169), (50.0, 150), (25.0, 131), (0.0, 114)]
PersuasionTrace        n=  353  top targets=[(99.75, 5), (50.0, 4), (65.0, 4), (90.25390625, 3), (99.0, 3), (69.75, 3), (49.5, 3), (49.0, 3)]
exclusions {'CMV post contains a delta trace': 183, 'duplicate CMV post; kept one episode': 1126}
CMV posts 2931 (one episode each, target always 100)


## No group appears in two splits

In [3]:
splits = {}
for ex in examples:
    splits.setdefault((ex.dataset, ex.group_id), set()).add(ex.split)
leaked = {key: value for key, value in splits.items() if len(value) > 1}
assert not leaked, leaked
print(f'{len(splits)} groups, each in exactly one split')

7209 groups, each in exactly one split


## Five examples

ConspiracyAI, DEBATE, and CMV include the participant's own initial text. AnthropicPersuasion and DebateGPT are proposition-only, so there is no assistant turn and the written argument is absent.

Change `EXAMPLE_INDEX` to print a different example from each dataset. `0` is the first.

In [8]:
EXAMPLE_INDEX = 10  # change this to view another example from each dataset

wanted = [
    ('ConspiracyAI', True),
    ('DEBATE', True),
    ('CMV', True),
    ('AnthropicPersuasion', False),
    ('DebateGPT', False),
]
picked = []
for dataset, needs_context in wanted:
    pool = [
        ex for ex in examples
        if ex.dataset == dataset and (ex.initial_context is not None) == needs_context
    ]
    if not 0 <= EXAMPLE_INDEX < len(pool):
        raise IndexError(f'{dataset} has {len(pool)} examples; EXAMPLE_INDEX={EXAMPLE_INDEX} is out of range')
    print(f'{dataset}: showing index {EXAMPLE_INDEX} of {len(pool)}')
    picked.append(pool[EXAMPLE_INDEX])
for ex in picked:
    print('=' * 24, ex.dataset, ex.episode_id, 'target', ex.target, ex.target_type)
    print(json.dumps(ex.messages, ensure_ascii=False, indent=2)[:3600])
    assert ex.messages[-1]['role'] == 'user'
    assert ex.messages[-1]['content'] == USER_QUESTION
    assert ex.proposition in ex.messages[0]['content']
    assert ex.dataset not in ex.messages[0]['content']
    assistants = [m for m in ex.messages if m['role'] == 'assistant']
    if ex.initial_context is None:
        assert assistants == []
    else:
        assert assistants[0]['content'] == ex.initial_context
print('message contract holds for the five examples')

ConspiracyAI: showing index 10 of 1749
DEBATE: showing index 10 of 1052
CMV: showing index 10 of 2931
AnthropicPersuasion: showing index 10 of 3939
DebateGPT: showing index 10 of 750
======================== ConspiracyAI conspiracyai:s1-2825:P1 target 91.0 continuous
[
  {
    "role": "system",
    "content": "You are a real person with your own opinions, speaking as yourself in the first person. You have a view on the following statement.\n\nStatement: The tobacco companies were aware of the health risks associated with smoking, including cancer, long before these risks were publicly disclosed, despite the significant financial repercussions acknowledging such information would have for the industry.\n\nAnswer as you genuinely feel, based on your own beliefs, experience and way of expressing yourself, not as you think you should."
  },
  {
    "role": "assistant",
    "content": "I believe that the tobacco companies know about the health risks of smoking long before they were made pub

## Ablation: drop the self-statement

`drop_context=True` leaves the system turn and the question. Shown here for one DEBATE participant.

In [5]:
ablation = InitialStanceDataLoader.from_config(
    REPO / 'configs' / 'datasets.yaml',
    repo_root=REPO,
    datasets=['DEBATE'],
    splits=['train'],
    max_examples=1,
    drop_context=True,
)
one = ablation.load()[0]
print(one.episode_id, 'initial_context', one.initial_context)
print([m['role'] for m in one.messages])
assert one.initial_context is None
assert [m['role'] for m in one.messages] == ['system', 'user']

debate:01JNP6Y2EBNR7HAY04ZBD4952J:P1 initial_context None
['system', 'user']


## Chat template

`ensure_alternation=True` inserts a neutral user turn (`Go ahead.`) before the first assistant message for templates that reject a system → assistant start. It is off by default. Pass the `messages` list to the tokenizer; do not put the target integer into the prompt.

In [6]:
from opinion_dynamics.tasks.initial_stance import build_messages
from opinion_dynamics.datasets.dataloader import OpinionDynamicsDataLoader

episode_loader = OpinionDynamicsDataLoader(repo_root=REPO, splits=['train'])
episode_loader.register('DEBATE', REPO / 'data' / 'unified' / 'debate.jsonl')
episode = next(ep for ep in episode_loader.load() if ep.agent.participant_id == 'P1')
plain = build_messages(episode, 'initial')
fixed = build_messages(episode, 'initial', ensure_alternation=True)
print('default roles', [m['role'] for m in plain])
print('alternation roles', [m['role'] for m in fixed])
assert fixed[1]['content'] == 'Go ahead.'
print('call: tokenizer.apply_chat_template(plain, add_generation_prompt=True)')

default roles ['system', 'assistant', 'user']
alternation roles ['system', 'user', 'assistant', 'user']
call: tokenizer.apply_chat_template(plain, add_generation_prompt=True)
